In [0]:
@dp.table(
  name="gold_agg_monthly_revenue_region",
  comment="Monthly revenue and order count by region"
)
def agg_monthly_revenue_region():

  fact = spark.readStream.table("gold_fact_transactions")
  dim_date = spark.readStream.table("gold_dim_date")

  return (
    fact.join(dim_date, "date_key")
    .groupBy(
      "year",
      "month",
      "region_key"
    )
    .agg(
      countDistinct("order_id").alias("order_count"),
      sum("sales_amount").alias("total_sales")
    )
  )

In [0]:
@dp.table(
  name="gold_agg_customer_returns",
  comment="Customer return count and value"
)
def agg_customer_returns():

  fact = spark.readStream.table("gold_fact_returns")

  return (
    fact.groupBy("customer_key")
    .agg(
      count("return_id").alias("return_count"),
      sum("return_amount").alias("total_return_value")
    )
  )

In [0]:
@dp.table(
  name="gold_agg_vendor_performance",
  comment="Vendor return rate and performance"
)
def agg_vendor_performance():

  sales = spark.readStream.table("gold_fact_transactions")
  returns = spark.readStream.table("gold_fact_returns")

  sales_agg = (
    sales.groupBy("vendor_key")
    .agg(
      count("transaction_id").alias("total_sales_txn"),
      sum("sales_amount").alias("total_sales")
    )
  )

  returns_agg = (
    returns.groupBy("vendor_key")
    .agg(
      count("return_id").alias("total_returns"),
      sum("return_amount").alias("total_return_value")
    )
  )

  return (
    sales_agg.join(returns_agg, "vendor_key", "left")
    .withColumn(
      "return_rate",
      col("total_returns") / col("total_sales_txn")
    )
  )

In [0]:
@dp.table(
  name="gold_agg_product_performance",
  comment="Product performance and slow moving indicators"
)
def agg_product_performance():

  sales = spark.readStream.table("gold_fact_transactions")
  returns = spark.readStream.table("gold_fact_returns")

  sales_agg = (
    sales.groupBy("product_key", "region_key")
    .agg(
      sum("sales_amount").alias("total_sales"),
      count("transaction_id").alias("sales_count")
    )
  )

  returns_agg = (
    returns.groupBy("product_key", "region_key")
    .agg(
      count("return_id").alias("return_count")
    )
  )

  return (
    sales_agg.join(returns_agg, ["product_key", "region_key"], "left")
    .withColumn(
      "return_rate",
      col("return_count") / col("sales_count")
    )
    .withColumn(
      "slow_moving_flag",
      when(col("sales_count") < 10, True).otherwise(False)
    )
  )